# **2.5. Load the Evidence Scrutinization Llama 3 model for Fine Tuning**

In [1]:
!nvidia-smi

Tue Mar  3 06:58:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -U bitsandbytes transformers peft accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 42.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, whi

In [3]:
# !pip install -U bitsandbytes>=0.46.1

In [4]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("HF_TOKEN")
import os
from kaggle_secrets import UserSecretsClient

# Get the secret from Kaggle and set it as an environment variable
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

**Load Dataset**

In [5]:
from datasets import load_dataset

# Load the TathyaNyaya dataset
dataset = load_dataset("L-NLProc/TathyaNyaya-and-FactLegalLlama-NyayaFilter-Datasets", split="train")

# Fetch the first record to see columns
first_record = next(iter(dataset))
print(f"Dataset Columns: {list(first_record.keys())}")


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

train_unique_new.csv:   0%|          | 0.00/510M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13629 [00:00<?, ? examples/s]

Dataset Columns: ['Unnamed: 0', 'Case Name', 'Date', 'Citation', 'Bench', 'Apellant Advocates', 'Respondent Advocates', 'Text', 'Facts', 'Issue', 'Reasoning', 'Arguments of Petitioner', 'Arguments of Respondent', 'Decision', 'Label']


**Prep Dataset**

In [6]:
def formatting_func(example):
    # This now includes the critical arguments for evidence scrutinization
    text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n" \
           f"You are an expert in Indian Law. Scrutinize the arguments and evidence to provide a reasoned legal analysis.<|eot_id|>" \
           f"<|start_header_id|>user<|end_header_id|>\n" \
           f"Case Name: {example['Case Name']}\n" \
           f"Facts: {example['Facts']}\n" \
           f"Issue: {example['Issue']}\n" \
           f"Petitioner's Arguments: {example['Arguments of Petitioner']}\n" \
           f"Respondent's Arguments: {example['Arguments of Respondent']}<|eot_id|>\n" \
           f"<|start_header_id|>assistant<|end_header_id|>\n" \
           f"Reasoning: {example['Reasoning']}\n" \
           f"Decision: {example['Decision']}<|eot_id|>"
    return {"text": text}

dataset = dataset.map(formatting_func)

Map:   0%|          | 0/13629 [00:00<?, ? examples/s]

**Load Model for Fine Tuning**

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

In [8]:
model_id = "unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit"

In [9]:
# 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [10]:
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Critical for Llama 3

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:250: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

In [11]:
# Configure LoRA - This creates the "Adapter"
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Target key layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

**Start Training & Save to Drive**

In [12]:
from trl import SFTTrainer,SFTConfig
# from transformers import TrainingArguments

In [17]:

sft_config = SFTConfig(
    output_dir="/kaggle/working/LPA/Nyaya_Finetune_Checkpoints",
    dataset_text_field="text",    
    max_length=2048,         
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=1,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_strategy="epoch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)

In [19]:

# Initialize the trainer with the config object
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

In [20]:
trainer.train()

Step,Training Loss
1,1.721773
2,1.907557
3,2.034032
4,1.874341
5,1.795088
6,1.737879
7,1.803579
8,1.896735
9,1.936359
10,1.787522


TrainOutput(global_step=60, training_loss=1.7555938720703126, metrics={'train_runtime': 2149.7442, 'train_samples_per_second': 0.112, 'train_steps_per_second': 0.028, 'total_flos': 1.92278051524608e+16, 'train_loss': 1.7555938720703126})

In [21]:
# Final Save: ONLY saves the small adapter (~100MB-200MB)
model.save_pretrained("/kaggle/working/LPA/Nyaya_Finetune_Checkpoints")

In [22]:
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Memory allocated: 1.56 GB
Memory reserved: 7.09 GB


In [23]:
import shutil
import os

# Define the source and destination
source_dir = "/kaggle/working/LPA/Nyaya_Finetune_Checkpoints"
zip_name = "Nyaya_Llama_Adapter_MTech"

# Create the zip file
shutil.make_archive(zip_name, 'zip', source_dir)

print(f"Success! Zip file created at: /kaggle/working/{zip_name}.zip")

Success! Zip file created at: /kaggle/working/Nyaya_Llama_Adapter_MTech.zip


**Old Prompts**